# 8-Puzzle Solver using A* Search

Implements the **A\* Algorithm** exactly as specified in the pseudocode from Lecture 3B, Slide 21, applied to the 8-puzzle with three heuristics:

1. **h1** — Number of Tiles in the Wrong Position
2. **h2** — Manhattan Distance
3. **h3** — Nilsson's Sequence Score: `h(n) = P(n) + 3*S(n)`

For each heuristic the notebook prints the full sequence of states from `start` to `goal` as 3x3 grids, with `g(n)`, `h(n)`, `f(n)` (and `P(n)`, `S(n)` for Nilsson) at every step, plus the search cost (total nodes generated).

Each of the 8 pseudocode steps is labeled `# Step k` in the `a_star` function below so it can be checked directly against the slide. Node records are kept in a dictionary keyed by state, giving O(1) "is this successor already on OPEN or CLOSED?" lookups — this is purely an efficiency choice and does not change the algorithm's logic or control flow.

## 1. Reading the input file

The puzzle is read from a text file in the required format (`*` = blank tile):

```
start
2 1 6
4 * 8
7 5 3
goal
1 2 3
8 * 4
7 6 5
```

Each grid is flattened row-major into a 9-element tuple (index 0 = top-left, index 8 = bottom-right).

In [1]:
# Write the sample puzzle to input.txt in the required format
# (Edit this cell, or point PUZZLE_FILE at another file, to test other puzzles.)

sample_input = """start
2 1 6
4 * 8
7 5 3
goal
1 2 3
8 * 4
7 6 5
"""

with open("input.txt", "w") as f:
    f.write(sample_input)

print(sample_input)


start
2 1 6
4 * 8
7 5 3
goal
1 2 3
8 * 4
7 6 5



In [2]:
def parse_input(filepath):
    """Reads the puzzle input file and returns (start, goal) as 9-tuples.
    Index layout (row-major):
        0 1 2
        3 4 5
        6 7 8
    """
    with open(filepath) as f:
        lines = [line.strip() for line in f if line.strip()]

    start_idx = lines.index("start")
    goal_idx = lines.index("goal")

    def parse_grid(grid_lines):
        tiles = []
        for row in grid_lines:
            tiles.extend(row.split())
        if len(tiles) != 9:
            raise ValueError(f"Expected 9 tiles, got {len(tiles)}: {tiles}")
        return tuple(tiles)

    start = parse_grid(lines[start_idx + 1: start_idx + 4])
    goal = parse_grid(lines[goal_idx + 1: goal_idx + 4])
    return start, goal


def print_state(state):
    """Pretty-prints a 9-tuple state as a 3x3 grid."""
    for r in range(3):
        row = state[r * 3:(r + 1) * 3]
        print(" ".join(row))


PUZZLE_FILE = "input.txt"
START, GOAL = parse_input(PUZZLE_FILE)

print("START:")
print_state(START)
print("\nGOAL:")
print_state(GOAL)


START:
2 1 6
4 * 8
7 5 3

GOAL:
1 2 3
8 * 4
7 6 5


## 2. Successor generation

Given a state, the blank (`*`) can move Up, Down, Left, or Right (whichever stay inside the 3x3 grid). Each resulting swap is one successor `n_i`, matching the "generate all of its successors" step of the pseudocode.

In [3]:
def get_successors(state):
    """Returns the list of states reachable by sliding a tile into the blank."""
    blank_idx = state.index("*")
    row, col = divmod(blank_idx, 3)
    successors = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:  # Up, Down, Left, Right
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_idx = nr * 3 + nc
            new_state = list(state)
            new_state[blank_idx], new_state[new_idx] = new_state[new_idx], new_state[blank_idx]
            successors.append(tuple(new_state))
    return successors


## 3. Heuristic functions

**a) Wrong Tiles** — number of tiles not in their goal position (blank excluded).

**b) Manhattan Distance** — sum, over every tile, of the taxicab distance between its current cell and its goal cell.

**c) Nilsson's Sequence Score** — `h(n) = P(n) + 3*S(n)`, where:
- `P(n)` is the Manhattan distance of each tile from its goal position, measured against **Nilsson's own fixed goal configuration** `1 2 3 / 8 * 4 / 7 6 5` — this is fixed regardless of whatever goal is in the input file, since the assignment specifies this as "the goal state that Nilsson uses."
- `S(n)` walks the 8 non-central squares clockwise starting top-left. For each square, add **2** if the tile there is *not* immediately followed (clockwise) by its proper successor (1→2→3→4→5→6→7→8→1) — a blank on a non-central square always scores 2. Add **1** if the center square is occupied by a tile (not blank).

In [4]:
def h_wrong_tiles(state, goal):
    """(a) Number of tiles in the wrong position (blank not counted)."""
    return sum(1 for s, g in zip(state, goal) if s != g and s != "*")


def h_manhattan(state, goal):
    """(b) Sum of Manhattan distances of every tile from its goal position."""
    goal_pos = {tile: idx for idx, tile in enumerate(goal)}
    total = 0
    for idx, tile in enumerate(state):
        if tile == "*":
            continue
        gr, gc = divmod(goal_pos[tile], 3)
        cr, cc = divmod(idx, 3)
        total += abs(gr - cr) + abs(gc - cc)
    return total


# --- Nilsson's Sequence Score ---

# Fixed reference goal Nilsson's heuristic is defined against, regardless of
# the puzzle's own goal file:
#   1 2 3
#   8 * 4
#   7 6 5
NILSSON_GOAL = ("1", "2", "3", "8", "*", "4", "7", "6", "5")

# Indices of the 8 non-central squares, walked clockwise starting top-left:
#   0 1 2
#   3 . 5
#   6 7 8
PERIMETER = [0, 1, 2, 5, 8, 7, 6, 3]

# Proper clockwise successor of each tile in Nilsson's goal sequence
SUCCESSOR = {"1": "2", "2": "3", "3": "4", "4": "5",
             "5": "6", "6": "7", "7": "8", "8": "1"}


def nilsson_P(state):
    """P(n): Manhattan distance of every tile from its position in NILSSON_GOAL."""
    return h_manhattan(state, NILSSON_GOAL)


def nilsson_S(state):
    """S(n): sequence score around the perimeter, plus 1 if the center is occupied."""
    s = 0
    n = len(PERIMETER)
    for i in range(n):
        pos = PERIMETER[i]
        next_pos = PERIMETER[(i + 1) % n]
        tile = state[pos]
        if tile == "*":
            s += 2
            continue
        if state[next_pos] != SUCCESSOR[tile]:
            s += 2
    if state[4] != "*":  # center square
        s += 1
    return s


def h_nilsson(state):
    """(c) h(n) = P(n) + 3*S(n). Returns (h, P, S)."""
    P = nilsson_P(state)
    S = nilsson_S(state)
    return P + 3 * S, P, S


def compute_h(state, goal, heuristic_name):
    """Dispatches to the requested heuristic. Returns (h_value, extra_dict)."""
    if heuristic_name == "wrong_tiles":
        return h_wrong_tiles(state, goal), {}
    elif heuristic_name == "manhattan":
        return h_manhattan(state, goal), {}
    elif heuristic_name == "nilsson":
        h_val, P, S = h_nilsson(state)
        return h_val, {"P": P, "S": S}
    else:
        raise ValueError(f"Unknown heuristic: {heuristic_name}")


## 4. A\* algorithm (follows the Slide 21 pseudocode step-by-step)

| Step | Pseudocode | Where in the code |
|---|---|---|
| 1 | Put start node `s` on OPEN, compute `f(s)` | node record created before the loop |
| 2 | If OPEN empty, exit with failure | `while True: if not OPEN: return ...` |
| 3 | Remove node with smallest `f` from OPEN onto CLOSED, call it `n` (ties → favor goal) | `min_f`, `candidates`, tie-break |
| 4 | If `n` is goal, exit with success (trace pointers) | `if n == goal: return ...` |
| 5 | Expand `n`, generate successors, compute `f(n_i)` for each | `get_successors`, `compute_h` loop |
| 6 | Successors not on OPEN/CLOSED → put on OPEN, point back to `n` | `if s not in nodes:` branch |
| 7 | Successors already on OPEN/CLOSED → keep smaller `f`; if lowered, move CLOSED→OPEN and redirect pointer to `n` | `else:` branch |
| 8 | Goto 2 | `while True` loop continues |

Node records live in a `dict` keyed by state, so "is this successor already on OPEN or CLOSED?" is an O(1) lookup — a pure efficiency choice that doesn't change the algorithm's behavior. `nodes_generated` counts every successor the algorithm computes `f(n_i)` for in Step 5 (Step 1's start node, plus one per successor considered — including successors that turn out to already exist on OPEN/CLOSED), which is reported as the **search cost**.

In [5]:
def a_star(start, goal, heuristic_name):
    """A* search following the Slide 21 pseudocode exactly.

    Returns:
        path             -- list of states from start to goal (None if no solution)
        nodes            -- dict of node records (state -> g, h, f, parent, P, S)
        nodes_generated  -- search cost: total number of successor f(n_i) computations
    """
    nodes = {}
    OPEN = []
    nodes_generated = 0

    # --- Step 1: put the start node s on OPEN and compute f(s) ---
    h0, extra0 = compute_h(start, goal, heuristic_name)
    nodes[start] = {"g": 0, "h": h0, "f": 0 + h0, "parent": None,
                     "status": "OPEN", "extra": extra0}
    OPEN.append(start)
    nodes_generated += 1

    while True:
        # --- Step 2: if OPEN is empty, exit with failure ---
        if not OPEN:
            return None, nodes, nodes_generated

        # --- Step 3: remove from OPEN the node with smallest f, put on CLOSED.
        #     Ties are resolved arbitrarily, but always in favor of a goal node. ---
        min_f = min(nodes[s]["f"] for s in OPEN)
        candidates = [s for s in OPEN if nodes[s]["f"] == min_f]
        n = goal if goal in candidates else candidates[0]
        OPEN.remove(n)
        nodes[n]["status"] = "CLOSED"

        # --- Step 4: if n is a goal node, exit with success (trace back pointers) ---
        if n == goal:
            path = []
            cur = n
            while cur is not None:
                path.append(cur)
                cur = nodes[cur]["parent"]
            path.reverse()
            return path, nodes, nodes_generated

        # --- Step 5: expand n, generate all successors, compute f(n_i) for each ---
        successors = get_successors(n)
        if not successors:
            continue  # "if there are no successors, go immediately to 2"

        g_n = nodes[n]["g"]
        for s in successors:
            h_val, extra = compute_h(s, goal, heuristic_name)   # compute f(n_i)
            g_new = g_n + 1                                     # uniform step cost 1
            f_new = g_new + h_val
            nodes_generated += 1

            if s not in nodes:
                # --- Step 6: not already on OPEN or CLOSED -> put on OPEN,
                #     pointer back to n ---
                nodes[s] = {"g": g_new, "h": h_val, "f": f_new, "parent": n,
                            "status": "OPEN", "extra": extra}
                OPEN.append(s)
            else:
                # --- Step 7: already on OPEN or CLOSED -> keep the smaller f.
                #     If lowered, and it was on CLOSED, move it back to OPEN
                #     and redirect its pointer to n. ---
                if f_new < nodes[s]["f"]:
                    was_closed = (nodes[s]["status"] == "CLOSED")
                    nodes[s]["g"] = g_new
                    nodes[s]["f"] = f_new
                    nodes[s]["parent"] = n
                    nodes[s]["extra"] = extra
                    if was_closed:
                        nodes[s]["status"] = "OPEN"
                        OPEN.append(s)
                # else: existing f is already smaller/equal -> keep it, do nothing
        # --- Step 8: go to 2 (loop repeats) ---


## 5. Reporting helper

Runs `a_star`, prints the resulting path as 3x3 grids with `f`, `g`, `h` (and `P`, `S` for Nilsson) at every step, and prints the search cost.

In [6]:
def run_heuristic(start, goal, heuristic_name, label):
    print("=" * 60)
    print(f"Heuristic: {label}")
    print("=" * 60)

    path, nodes, nodes_generated = a_star(start, goal, heuristic_name)

    if path is None:
        print("No solution found.")
        print(f"Search cost (nodes generated): {nodes_generated}")
        return nodes_generated

    for step, state in enumerate(path):
        info = nodes[state]
        print(f"--- Step {step} ---")
        print_state(state)
        if info["extra"]:
            extras = ", ".join(f"{k}(n)={v}" for k, v in info["extra"].items())
            print(f"g(n)={info['g']}  h(n)={info['h']}  f(n)={info['f']}  {extras}")
        else:
            print(f"g(n)={info['g']}  h(n)={info['h']}  f(n)={info['f']}")
        print()

    print(f"Solution length (moves): {len(path) - 1}")
    print(f"Search cost (nodes generated): {nodes_generated}")
    print()
    return nodes_generated


## 6. Run all three heuristics on the sample puzzle

In [7]:
cost_wrong_tiles = run_heuristic(START, GOAL, "wrong_tiles",
                                  "(a) Number of Tiles in the Wrong Position")


Heuristic: (a) Number of Tiles in the Wrong Position
--- Step 0 ---
2 1 6
4 * 8
7 5 3
g(n)=0  h(n)=7  f(n)=7

--- Step 1 ---
2 1 6
4 8 *
7 5 3
g(n)=1  h(n)=7  f(n)=8

--- Step 2 ---
2 1 *
4 8 6
7 5 3
g(n)=2  h(n)=7  f(n)=9

--- Step 3 ---
2 * 1
4 8 6
7 5 3
g(n)=3  h(n)=7  f(n)=10

--- Step 4 ---
2 8 1
4 * 6
7 5 3
g(n)=4  h(n)=7  f(n)=11

--- Step 5 ---
2 8 1
4 6 *
7 5 3
g(n)=5  h(n)=7  f(n)=12

--- Step 6 ---
2 8 1
4 6 3
7 5 *
g(n)=6  h(n)=7  f(n)=13

--- Step 7 ---
2 8 1
4 6 3
7 * 5
g(n)=7  h(n)=6  f(n)=13

--- Step 8 ---
2 8 1
4 * 3
7 6 5
g(n)=8  h(n)=5  f(n)=13

--- Step 9 ---
2 8 1
* 4 3
7 6 5
g(n)=9  h(n)=5  f(n)=14

--- Step 10 ---
* 8 1
2 4 3
7 6 5
g(n)=10  h(n)=5  f(n)=15

--- Step 11 ---
8 * 1
2 4 3
7 6 5
g(n)=11  h(n)=5  f(n)=16

--- Step 12 ---
8 1 *
2 4 3
7 6 5
g(n)=12  h(n)=5  f(n)=17

--- Step 13 ---
8 1 3
2 4 *
7 6 5
g(n)=13  h(n)=4  f(n)=17

--- Step 14 ---
8 1 3
2 * 4
7 6 5
g(n)=14  h(n)=3  f(n)=17

--- Step 15 ---
8 1 3
* 2 4
7 6 5
g(n)=15  h(n)=3  f(n)=18

--- Step 1

In [8]:
cost_manhattan = run_heuristic(START, GOAL, "manhattan",
                                "(b) Manhattan Distance")


Heuristic: (b) Manhattan Distance
--- Step 0 ---
2 1 6
4 * 8
7 5 3
g(n)=0  h(n)=12  f(n)=12

--- Step 1 ---
2 1 6
4 8 *
7 5 3
g(n)=1  h(n)=11  f(n)=12

--- Step 2 ---
2 1 *
4 8 6
7 5 3
g(n)=2  h(n)=10  f(n)=12

--- Step 3 ---
2 * 1
4 8 6
7 5 3
g(n)=3  h(n)=11  f(n)=14

--- Step 4 ---
2 8 1
4 * 6
7 5 3
g(n)=4  h(n)=12  f(n)=16

--- Step 5 ---
2 8 1
4 6 *
7 5 3
g(n)=5  h(n)=11  f(n)=16

--- Step 6 ---
2 8 1
4 6 3
7 5 *
g(n)=6  h(n)=10  f(n)=16

--- Step 7 ---
2 8 1
4 6 3
7 * 5
g(n)=7  h(n)=9  f(n)=16

--- Step 8 ---
2 8 1
4 * 3
7 6 5
g(n)=8  h(n)=8  f(n)=16

--- Step 9 ---
2 8 1
* 4 3
7 6 5
g(n)=9  h(n)=7  f(n)=16

--- Step 10 ---
* 8 1
2 4 3
7 6 5
g(n)=10  h(n)=8  f(n)=18

--- Step 11 ---
8 * 1
2 4 3
7 6 5
g(n)=11  h(n)=7  f(n)=18

--- Step 12 ---
8 1 *
2 4 3
7 6 5
g(n)=12  h(n)=6  f(n)=18

--- Step 13 ---
8 1 3
2 4 *
7 6 5
g(n)=13  h(n)=5  f(n)=18

--- Step 14 ---
8 1 3
2 * 4
7 6 5
g(n)=14  h(n)=4  f(n)=18

--- Step 15 ---
8 1 3
* 2 4
7 6 5
g(n)=15  h(n)=3  f(n)=18

--- Step 16 ---
* 1

In [9]:
cost_nilsson = run_heuristic(START, GOAL, "nilsson",
                              "(c) Nilsson's Sequence Score")


Heuristic: (c) Nilsson's Sequence Score
--- Step 0 ---
2 1 6
4 * 8
7 5 3
g(n)=0  h(n)=60  f(n)=60  P(n)=12, S(n)=16

--- Step 1 ---
2 1 6
4 8 *
7 5 3
g(n)=1  h(n)=62  f(n)=63  P(n)=11, S(n)=17

--- Step 2 ---
2 1 *
4 8 6
7 5 3
g(n)=2  h(n)=61  f(n)=63  P(n)=10, S(n)=17

--- Step 3 ---
2 * 1
4 8 6
7 5 3
g(n)=3  h(n)=62  f(n)=65  P(n)=11, S(n)=17

--- Step 4 ---
2 8 1
4 * 6
7 5 3
g(n)=4  h(n)=54  f(n)=58  P(n)=12, S(n)=14

--- Step 5 ---
2 8 1
4 6 *
7 5 3
g(n)=5  h(n)=56  f(n)=61  P(n)=11, S(n)=15

--- Step 6 ---
2 8 1
4 6 3
7 5 *
g(n)=6  h(n)=55  f(n)=61  P(n)=10, S(n)=15

--- Step 7 ---
2 8 1
4 6 3
7 * 5
g(n)=7  h(n)=54  f(n)=61  P(n)=9, S(n)=15

--- Step 8 ---
2 8 1
4 * 3
7 6 5
g(n)=8  h(n)=38  f(n)=46  P(n)=8, S(n)=10

--- Step 9 ---
2 8 1
* 4 3
7 6 5
g(n)=9  h(n)=40  f(n)=49  P(n)=7, S(n)=11

--- Step 10 ---
* 8 1
2 4 3
7 6 5
g(n)=10  h(n)=41  f(n)=51  P(n)=8, S(n)=11

--- Step 11 ---
8 * 1
2 4 3
7 6 5
g(n)=11  h(n)=46  f(n)=57  P(n)=7, S(n)=13

--- Step 12 ---
8 1 *
2 4 3
7 6 5
g(n

## 7. Search cost summary

In [10]:
print(f"{'Heuristic':40s} {'Search cost (nodes generated)':>30s}")
print(f"{'Number of Tiles in the Wrong Position':40s} {cost_wrong_tiles:>30d}")
print(f"{'Manhattan Distance':40s} {cost_manhattan:>30d}")
print(f"{'Nilsson Sequence Score':40s} {cost_nilsson:>30d}")


Heuristic                                 Search cost (nodes generated)
Number of Tiles in the Wrong Position                              4777
Manhattan Distance                                                  616
Nilsson Sequence Score                                               98
